# Export a Q# circuit to SVG

This notebook uses the existing `qdk.widgets.Circuit` host. The notebook view and **Export as SVG** button both use QDK's shared TypeScript circuit renderer, so there is no notebook-specific renderer, DOM snapshot, Node service, or Quantikz dependency.

## 1. Import QDK

Install the optional `qdk[jupyter]` extra so the existing QDK circuit widget is available.

In [ ]:
import json

from qdk import qsharp
from qdk.widgets import Circuit

## 2. Define a small publication example

The parameterized `Rx` gate exercises the circuit-font path that a mounted-DOM prototype could accidentally inherit from its host.

In [ ]:
qsharp.eval(
    """
    operation PublicationCircuit(angle : Double) : Result[] {
        use qubits = Qubit[2];
        H(qubits[0]);
        CNOT(qubits[0], qubits[1]);
        Rx(angle, qubits[1]);
        let results = MeasureEachZ(qubits);
        ResetAll(qubits);
        return results;
    }
    """
)

## 3. Generate and check the circuit model

`qsharp.circuit()` returns semantic circuit data. It is not itself a widget or an SVG.

In [ ]:
circuit = qsharp.circuit("PublicationCircuit(0.7853981633974483)")
circuit_data = json.loads(circuit.json())

assert len(circuit_data["qubits"]) == 2
assert circuit_data["componentGrid"]

## 4. Display the existing QDK circuit widget

The widget passes the circuit JSON to the same shared TypeScript `Circuit` component used by QDK's VS Code views.

In [ ]:
widget = Circuit(circuit)
widget

## 5. Export the circuit

Select **Export as SVG** above the circuit. The shared TypeScript renderer generates a standalone SVG directly from the circuit model and the current expansion state, then the notebook frontend downloads it.

Phase 1 intentionally does not add `await widget.to_svg()`: returning SVG text to Python requires a separate asynchronous Jupyter communication protocol. The button and that future Python method would use the same SVG renderer.

## 6. Verify the exported file

Open the saved SVG in a browser or vector editor. It should remain sharp at any zoom, retain the QDK circuit geometry, use embedded QDK-owned KaTeX font assets, and have a transparent outer canvas with explicit gate fills and strokes.

## 7. Use the SVG in Overleaf

Upload the SVG to your Overleaf project and include it with the `svg` package:

```latex
\includesvg[
  width=\linewidth,
  inkscapelatex=false
]{publication-circuit}
```

Setting `inkscapelatex=false` keeps the circuit text and embedded fonts inside the SVG. The default LaTeX-text mode replaces that text and can change the QDK appearance.